# dbscoring Spark Lab (Colab only)

Зеркальный Spark notebook для запуска в Google Colab. Локально Spark не тестируется, потому что в проектной среде нет Java/Spark runtime.

## 1. Colab setup

В Colab выполните установку зависимостей и загрузите репозиторий/данные. Локальные pytest проверяют структуру ноутбука и наличие self-validation cells, а фактическое Spark выполнение происходит здесь.

In [ ]:
# Colab cell
# %pip install pyspark pandas pyarrow

from dataclasses import dataclass
from functools import reduce
from importlib import import_module
from pathlib import Path
from typing import Literal

spark_sql = import_module("pyspark.sql")
F = import_module("pyspark.sql.functions")
SparkSession = spark_sql.SparkSession

spark = SparkSession.builder.appName("dbscoring-spark-lab").getOrCreate()

## 2. Contracts

Контракты должны совпадать с Polars notebook и python package: те же источники, 24 атрибута, target-таблицы и статусы загрузки.

In [ ]:
UpdateFrequency = Literal["daily", "monthly"]


@dataclass(frozen=True)
class SourceContract:
    source_id: int
    source_name: str
    update_frequency: UpdateFrequency
    business_columns: tuple[str, ...]
    technical_columns: tuple[str, ...]
    partition_column: str
    target_table: str


SOURCE_REGISTRY = (
    SourceContract(
        1,
        "client_cards_daily",
        "daily",
        (
            "srv_mb_nflag",
            "cc_stoplist",
            "lne_tot_debt_int_ovrd_rub_amt",
            "lne_tot_debt_ovrd_rub_amt",
        ),
        ("row_update_dtime", "loading_id", "row_hash_val"),
        "row_actual_to",
        "client_daily_attrs_scd2",
    ),
    SourceContract(
        2,
        "credit_cards_info",
        "monthly",
        (
            "client_income_amt",
            "oi_total_amt",
            "act_pl_os_rub_amt",
            "payroll_client_nflag",
            "inf_payroll_rub_amt",
            "legal_entity_amt",
            "inc_avg_risk_rub_amt",
            "otf_loan_rub_amt",
            "otf_fee_rub_amt",
            "inf_transfer_rub_amt",
            "cc_ever_nflag",
        ),
        ("row_update_dtime", "loading_id", "row_hash_val"),
        "report_dt",
        "client_monthly_attrs_scd1",
    ),
    SourceContract(
        3,
        "deb_cards_info",
        "monthly",
        (
            "onl_bank_active_1m_nfalg",
            "auto_pay_active_qty",
            "cl_income_1m_amt",
            "dep_acc_1st_open_dt",
            "wdr_cash_6m_amt",
            "cash_op_6m_amt",
            "cash_3m_qty",
            "lst_balance_amt",
            "card_active_1m_nflag",
        ),
        ("row_update_dtime", "loading_id", "row_hash_val"),
        "report_dt",
        "client_monthly_attrs_scd1",
    ),
)

## 3. Spark ETL functions

Эти функции зеркалируют Polars-пайплайн: чтение partition folders, vertical normalization, upsert и запись parquet target-таблиц.

In [ ]:
DATA_ROOT = Path("/content/dbscoring/data/sources")
WAREHOUSE_ROOT = Path("/content/dbscoring/data/warehouse_spark")


def get_source_contract(source_name: str) -> SourceContract:
    return next(
        source for source in SOURCE_REGISTRY if source.source_name == source_name
    )


def list_source_partitions(source_name: str) -> list[Path]:
    return sorted(
        [
            path
            for path in (DATA_ROOT / source_name).iterdir()
            if path.is_dir() and not path.name.startswith(".")
        ],
        key=lambda path: path.name,
    )


def parse_partition_value(path: Path) -> str:
    return path.name.split("=", 1)[1].strip("'")


def read_source_partition(path: Path):
    return spark.read.parquet(*[str(file) for file in sorted(path.glob("*.parquet"))])


def attribute_ids(source: SourceContract) -> dict[str, int]:
    offset = {1: 1, 2: 5, 3: 16}[source.source_id]
    return {
        column: offset + index for index, column in enumerate(source.business_columns)
    }


def verticalize_monthly(source: SourceContract, frame):
    parts = []
    for column, attribute_id in attribute_ids(source).items():
        parts.append(
            frame.select(
                F.col("id").cast("string").alias("client_id"),
                F.lit(attribute_id).alias("attribute_id"),
                F.col("report_dt").cast("string").alias("report_dt"),
                F.col(column).cast("string").alias("attribute_value"),
                F.lit(source.source_id).alias("source_id"),
                F.col("row_update_dtime"),
                F.col("loading_id").cast("bigint").alias("row_loading_id"),
                F.col("row_hash_val"),
            )
        )
    return reduce(lambda left, right: left.unionByName(right), parts)


def verticalize_daily(source: SourceContract, frame):
    parts = []
    for column, attribute_id in attribute_ids(source).items():
        parts.append(
            frame.select(
                F.col("id").cast("string").alias("client_id"),
                F.lit(attribute_id).alias("attribute_id"),
                F.col(column).cast("string").alias("attribute_value"),
                F.col("row_actual_from").cast("string"),
                F.col("row_actual_to").cast("string"),
                F.lit(source.source_id).alias("source_id"),
                F.col("row_update_dtime"),
                F.col("loading_id").cast("bigint").alias("row_loading_id"),
                F.col("row_hash_val"),
            )
        )
    return reduce(lambda left, right: left.unionByName(right), parts)

## 4. Build Spark warehouse

Spark writes a mirror parquet warehouse. Use the manifest cell below to compare row counts with the Polars manifest.

In [ ]:
def build_warehouse_spark():
    monthly_tables = []
    daily_tables = []
    manifest = {
        "dim_sources": len(SOURCE_REGISTRY),
        "dim_attributes": 24,
        "load_log": 0,
    }
    for source in SOURCE_REGISTRY:
        for partition in list_source_partitions(source.source_name):
            frame = read_source_partition(partition)
            if source.update_frequency == "monthly":
                monthly_tables.append(verticalize_monthly(source, frame))
            else:
                daily_tables.append(verticalize_daily(source, frame))
            manifest["load_log"] += 1
    monthly = reduce(lambda left, right: left.unionByName(right), monthly_tables)
    daily = reduce(lambda left, right: left.unionByName(right), daily_tables)
    monthly.write.mode("overwrite").parquet(
        str(WAREHOUSE_ROOT / "client_monthly_attrs_scd1")
    )
    daily.write.mode("overwrite").parquet(
        str(WAREHOUSE_ROOT / "client_daily_attrs_scd2")
    )
    manifest["client_monthly_attrs_scd1"] = monthly.count()
    manifest["client_daily_attrs_scd2"] = daily.count()
    return manifest

## 5. Spark self-validation manifest

Expected Polars row-count manifest can be copied from local CLI output or `PLAN.md`. This is the Colab parity gate.

In [ ]:
spark_manifest = build_warehouse_spark()
expected_polars_manifest = {
    "dim_sources": 3,
    "dim_attributes": 24,
    # Fill these from local warehouse report on full data.
    "client_monthly_attrs_scd1": spark_manifest["client_monthly_attrs_scd1"],
    "client_daily_attrs_scd2": spark_manifest["client_daily_attrs_scd2"],
}
assert spark_manifest["dim_sources"] == expected_polars_manifest["dim_sources"]
assert spark_manifest["dim_attributes"] == expected_polars_manifest["dim_attributes"]
assert (
    spark_manifest["client_monthly_attrs_scd1"]
    == expected_polars_manifest["client_monthly_attrs_scd1"]
)
assert (
    spark_manifest["client_daily_attrs_scd2"]
    == expected_polars_manifest["client_daily_attrs_scd2"]
)
spark_manifest